# 工程化与总结

## Multi-Agent Frameworks — AutoGen 与 CrewAI 实战指南

协作优化 · 最佳实践 · 常见陷阱 · 课程总结


## 📖 讲义
### 课程资料

2.4 Agents 协作与通信优化
通信模式：同步 RPC（适实时决策） vs 异步消息队列（适扩展/耐故障）。

协议设计：定义清晰的消息 schema（input/output）、错误码、幂等键，方便重试与回放。

节奏控制：为不同任务使用不同优先级与队列（realtime vs batch），避免关键路径被批处理阻塞。

信息压缩：在 Agent 间传递摘要或引用（如 doc#page#chunk），而非整文，节省带宽/延迟。

2.5 Agent 监督（Supervision）与可观测性
Trace/Run：记录 Thought/Action/Observation 或 Agent 的每次决策与工具调用，便于回放与审计。

质量门 (Quality Gates)：在关键阶段（例如最终答案生成前）插入 verifier Agent（NLI/检索复核）或 HITL（人工复核）。

监控指标：latency、error rate、queue length、task success rate、cost/token、verifier pass rate。

治理：权限/审计/数据保留策略，控制 Agent 能访问的外部资源。

4. 最佳实践 / 常见陷阱
最佳实践
职责分解明确：把复杂能力拆为小而可测试的 Agent（单一职责）。

消息 schema & 版本化：任务消息采用 JSON schema，带 schema_version 以支持逐步演进。

幂等与重试：设计幂等 key（job_id + task_id），确保重试安全。

Tracing 与可回放：记录每次 Agent 的输入/输出（prompt/retrieved/outputs），便于回放与评估。

质量门与 HITL：自动化 verifier + 人工复核组合，关键场景不可完全自动化。

分层扩展：不同层按需扩容（Retriever 高并发、Generator 成本高谨慎扩容）。

安全与最小权限：严格控制 Agent 能访问的外部系统与数据。

成本感知：对高成本 Agent（大模型）做预算与降级策略（缓存、小模型预筛）。

测试与模拟：先在沙箱做大量模拟（异常、延迟、失败），验证降级与恢复路径。

持续评估：用 RAGAS 等指标监控合成质量并做回归测试。

常见陷阱
没有停止条件的循环：Agent 间可能互相触发导致循环，应有步数/成本/时间上限。

过度中心化 Orchestrator：单点失效或性能瓶颈。可考虑分片 Orchestrator 或队列分散负载。

高基数日志导致成本暴涨：对 trace 做采样，异常/错误做全量记录。

共享可变状态导致竞态：使用事务或乐观锁避免竞态。

直接暴露危险能力（shell/数据库写入）给 Agent 而无审计或白名单。

忽视隐私/合规：多 Agent 共享数据时需做脱敏与访问控制。

---

---

### 幻灯片

# Multi-Agent Frameworks
## AutoGen 与 CrewAI 实战指南

课程学习资料 · 面向零基础 AI Engineer 训练营

含 LangGraph 对比与选型 · 120 分钟 · 88 页

---

## 五、协作与通信优化

- 多 Agent 的瓶颈常常不在模型本身
- 真正卡住系统的，常常是通信、调度和状态管理
- 因此要像做分布式系统一样做协作设计
- 不同框架只是入口不同，工程约束并不会消失

---

## 同步 vs 异步

- 同步 RPC：适合必须立刻返回的步骤
- 异步消息队列：适合需要缓冲和扩展的步骤
- 显式状态推进：适合分支多、需恢复的流程
- 混合架构最常见，不要期待一种方式通吃

建议：

- 关键路径短小同步
- 大吞吐和不稳定步骤异步

---

## Schema：消息、任务与状态

- 给协作载体定义固定结构
- 可以是 message schema、task contract 或 state schema
- 记录 input、output、error、schema_version
- 让不同 Agent 之间“说同一种语言”
- schema 是重试、回放和审计的前提

示例字段：

- job_id
- task_id
- payload
- retry_count
- status

---

## 幂等与重试

- 同一个任务重发时不应产生重复副作用
- 通过 job_id + task_id 做幂等键
- 失败后按策略重试，而不是盲目重复执行
- 重试前先判断是临时错误还是逻辑错误
- 把重试设计进系统，而不是事故后补救

常见重试策略：

- 立即重试
- 指数退避
- 最大次数上限

---

## 优先级与反压

- realtime 队列不要被 batch 任务堵住
- 高优先级任务要有独立通道
- 反压可以防止系统把自己压垮
- 预算和速率限制也属于反压的一部分
- 没有反压，系统往往越忙越差

如果没有反压：

- 队列会越堆越长
- 延迟会越来越大
- 成本会失控

---

## 信息压缩

- Agent 间传摘要，不传整篇原文
- 传引用，不传整份上下文
- 传 chunk id，不传整个文档
- 只把下一步真正需要的内容送过去
- 这和选大上下文模型不是一回事

这样可以：

- 降低 token
- 降低延迟
- 降低上下文污染

---

## Trace 与回放

- Trace 记录每一步输入和输出
- 回放帮助我们复现问题
- 审计帮助我们理解系统为何做出某个决策
- 对多 Agent 来说，trace 是最关键的调试材料
- 要能从 trace 看出“卡在哪一步”

推荐记录：

- prompt
- retrieved context
- tool calls
- model output
- final decision

---

## 监控指标

- latency
- error rate
- queue length
- task success rate
- cost/token
- verifier pass rate

> 讲师提示：让同学记住“系统变差时，先看指标，不要先怪模型”。

---

## 质量门

- 在最终输出前设置 verifier
- verifier 可以做检索复核、NLI 检验或规则检查
- 关键场景不通过就不能直接放行
- 质量门应当是流程的一部分，而不是补丁
- 它让系统拥有“可控失败”的出口

质量门的价值：

- 提前拦截低质量结果
- 给系统增加可控的失败出口

---

## 人工复核

- 当自动验证不确定时，交给人
- HITL 不是倒退，而是质量策略的一部分
- 人工结果也可以反哺后续优化
- 真正高风险任务不应默认全自动放行
- 人类介入点越清楚，系统越可治理

适合 HITL 的场景：

- 高风险内容
- 低置信度结果
- 涉及合规或业务决策

---

## 安全与治理

- Agent 只能访问必要资源
- 危险能力必须白名单和审计
- 需要考虑隐私、保留和合规
- 工具权限要最小化，不要默认全开
- 数据流向要可解释、可追踪、可回收

不要把以下能力随便开放给 Agent：

- shell
- 数据库写权限
- 外部 API 的高权限密钥

---

## Agentic RAG 总体图

- Retriever：找候选资料
- Reranker：精排候选
- Generator：生成答案
- Verifier：检查关键断言
- Orchestrator：决定放行或重试

这就是最常见的多 Agent RAG 骨架。

---

## 示例 A 概览

- AutoGen 风格三角色流水线
- 角色：Researcher -> Writer -> Editor
- 适合课程里快速展示“角色分工”
- 非常适合做第一轮教学原型

教学目标：

- 看懂协作链
- 理解上下文传递
- 理解审校环节为什么重要

---

## 示例 A 伪代码

```python
researcher = Agent(name="Researcher", capability="search_and_extract")
writer = Agent(name="Writer", capability="compose_summary")
editor = Agent(name="Editor", capability="fact_check_and_edit")

def orchestrate(topic):
    docs = researcher.run({"task": "search", "query": topic})
    draft = writer.run({"task": "write", "context": docs})
    final = editor.run({"task": "verify_and_edit", "draft": draft, "context": docs})
    return final
```

---

## 示例 A 教学点

- Researcher 只负责找资料
- Writer 只负责组织表达
- Editor 只负责核查和改写
- 每个 Agent 的边界要清楚
- 建模提示：AutoGen 最自然，CrewAI 可改写为 3 个 Task，LangGraph 可改成 3 个顺序节点 + 共享 state

> 课堂提问：如果 Writer 直接去检索，会发生什么？

---

## 示例 B 概览

- CrewAI / 队列风格的编队模式
- Orchestrator 把任务切成子任务
- Worker 通过队列逐步处理
- 适合高并发、长流程和可扩容任务链

适合：

- 高并发
- 长流程
- 可扩容任务链

---

## 示例 B 伪代码

```python
job_id = create_job(topic)
push_queue("retriever", {"job_id": job_id, "topic": topic})

# retriever worker
# reranker worker
# generator worker
# result db
```

---

## 示例 B 工程点

- 每个队列都能水平扩容 Worker
- job_id 用于结果聚合
- 队列长度能直接反映系统压力
- worker 健康检查决定是否自动重试
- 建模提示：CrewAI 最贴近任务流，AutoGen 更像前置原型，LangGraph 可用于队列内关键决策子流程

这是更接近生产系统的组织方式。

---

## 示例 C 概览

- Agentic RAG 的多 Agent 协作
- Retriever、Reranker、Generator、Verifier 分工明确
- Orchestrator 负责最终放行
- 非常适合讲“协作 + 监督”

这是最适合讲“协作 + 监督”两件事的例子。

---

## 示例 C 伪代码

```python
retrieval_job = dispatch("retriever", {"query": q, "top_k": 50})
candidates = wait_for_result(retrieval_job)
top5 = call_agent("reranker", {"query": q, "candidates": candidates})
draft = call_agent("generator", {"query": q, "context": top5})
verify = call_agent("verifier", {"draft": draft, "context": top5})
```

---

## 示例 C 的验证路径

```python
if verify["pass"]:
    return draft
else:
    send_to_human_review(draft, verify)
```

- pass 走自动放行
- fail 走人工复核或重试
- 建模提示：LangGraph 很适合表达 verify 分支，CrewAI 适合任务链，AutoGen 适合先演示协作与审校

---

## 七、最佳实践

- 最佳实践不是“写更多代码”
- 最佳实践是“让系统更稳、更可解释、更可扩展”
- 这些原则对 AutoGen、CrewAI、LangGraph 都成立

---

## 最佳实践 1：职责分解与抽象匹配

- 职责分解要明确，单一职责优先
- 如果一个 Agent 做太多事，调试会很难
- 要让框架抽象和问题结构匹配
- 对话问题用对话抽象，任务问题用任务抽象，状态问题用状态抽象

如果一个 Agent 做太多事：

- 调试会变难
- 重试会变危险
- 扩展会变贵

---

## 最佳实践 2：Schema 与版本化

- 消息 schema、task contract、state schema 都要版本化
- 兼容旧结构，允许逐步演进
- 让不同阶段的 Agent 可以平滑升级
- 清晰的 schema 是多 Agent 协作的公共语言

推荐字段：

- schema_version
- job_id
- task_id
- payload
- trace_id

---

## 最佳实践 3：幂等、Trace 与审计

- 幂等 key 必须设计好
- 失败重试必须可控
- trace 要能回放
- 关键步骤要可审计
- 把这四件事一起做，系统才有生产基础

这四件事决定系统能不能进入生产。

---

## 最佳实践 4：先模拟异常，再接真实世界

- 先在沙箱里模拟异常
- 再看降级和恢复是否有效
- 再做真实数据和真实工具接入
- 再考虑规模化扩容
- 测试顺序比“功能总数”更重要

测试顺序很重要：

- 正常路径
- 失败路径
- 边界路径
- 恢复路径

---

## 最佳实践 5：预算、成本与长期演进

- 预算和成本要前置考虑
- 高成本 Agent 要有降级方案
- 可以先用小模型预筛，再用大模型精处理
- 框架也要考虑迁移路径，不要把自己锁死
- 把贵的能力用在最值钱的地方

原则：

- 先证明价值，再扩大规模

---

## 八、常见陷阱

- 多 Agent 系统常见的问题，往往不是“不会调用 API”
- 而是“没有把失败设计进去”
- 还有一个隐藏问题：选错抽象层，会把简单问题做复杂

---

## 陷阱 1：没有停止条件的循环

- Agent 之间互相触发，越跑越久
- 最后成本和延迟都失控
- 对话驱动框架里，这个问题尤其常见

解决方式：

- 步数上限
- 时间上限
- 成本上限

---

## 陷阱 2：过度中心化的 Orchestrator

- 所有逻辑都压在一个点上
- 容易成为瓶颈或单点故障
- 也会让系统演进越来越困难

改进方向：

- 分片
- 队列化
- 事件驱动

---

## 陷阱 3：高基数日志导致成本暴涨

- 每个细节都全量记录会很贵
- trace 太多也会拖慢系统
- 观测不是越多越好，而是越有用越好

建议：

- 常规采样
- 异常全量
- 关键链路保留

---

## 陷阱 4：共享可变状态导致竞态

- 两个 Agent 同时改同一份状态
- 最后结果互相覆盖
- 状态驱动框架里尤其要小心写入边界

解决方式：

- 事务
- 乐观锁
- 明确的写入权责

---

## 陷阱 5：直接把危险能力交给 Agent

- 没有审计，没有白名单
- 一旦出错，影响会被放大
- 不要把“能调用”误当成“应该调用”

至少要有：

- 权限隔离
- 操作审计
- 人工兜底

---

## 陷阱 6：忽视隐私、合规与选型边界

- 多 Agent 共享数据时没有脱敏
- 结果是“功能跑通了，生产不能上”
- 还有一种问题是：为炫技而上复杂框架

生产前先问自己：

- 这份数据能不能传
- 这个结果能不能留
- 这个动作能不能自动执行

> 📖 完整讲义已嵌入本节（原 `课程资料.md` / `Marp 幻灯片.md` 已删除）


## 📖 讲义
### 课程资料

6. 小结（Summary）
Multi-Agent Frameworks（AutoGen、CrewAI） 为复杂任务提供模块化、可扩展且易维护的实现路径。

关键能力：职责拆分、消息/事件协作、幂等与重试、trace 与监督、质量门与 HITL。

实现路径：先做 AutoGen 风格的原型快速验证角色与流程，再用 CrewAI / 编队 + 消息队列做生产级扩展与调度。

工程注意：安全、可观测性、成本控制与可回放是投入生产的前提。

课程描述
学习目标

- 理解Multi-Agent Frameworks的核心概念，并学习如何使用AutoGen和CrewA构建多Agent系统

- 掌握如何通过这些框架设计、部署和管理多个Agents，处理复杂任务

- 学习如何优化Agents之间的协作与通信，提升系统的任务执行效率

- 探索如何在实际项目中使用AutoGen和CrewA框架进行多任务调度和自动化操作

知识点介绍

**Multi-Agent Frameworks概述**

- 了解Multi-Agent Systems (MAS)的基础架构，理解AutoGen和CrewA在多Agent系统中的作用

- 学习如何通过这些框架实现任务的自动化与并行处理

**AutoGen Framework的使用**

- 掌握AutoGen框架的核心功能和模块，学习如何使用它来自动生成和管理多个Agents

- 理解如何通过AutoGen实现动态任务分配与自动化调度

**CrewA Framework的应用**

- 掌握CrewA的功能特性，了解如何使用CrewA进行多Agent系统的协作与通信

- 学习如何通过CrewA优化任务执行顺序与资源分配，确保系统的高效运行

**Agents的协作与通信优化**

- 研究如何在复杂任务环境中设计Agents之间的协作机制，确保任务的顺利执行

- 学习如何通过这些框架提升Agents之间的通信效率与任务协调

**实际应用场景与系统扩展**

- 探讨Multi-Agent Frameworks在自动化控制、智能决策、生成式AI等领域的应用

- 学习如何扩展AutoGen和CrewA系统的规模与处理能力，以适应更复杂的任务环境

通过这些知识点，学员将能够使用AutoGen和CrewA构建、管理并优化多Agent系统，确保系统在实际项目中的高效运行和稳定性。

---

### 幻灯片

# Multi-Agent Frameworks
## AutoGen 与 CrewAI 实战指南

课程学习资料 · 面向零基础 AI Engineer 训练营

含 LangGraph 对比与选型 · 120 分钟 · 88 页

---

## 十、总结回顾

- Multi-Agent Frameworks 的核心是职责拆分与协作
- AutoGen 适合快速原型和教学演示
- CrewAI 更适合长期执行与任务编排
- LangGraph 更适合状态图工作流与可恢复流程
- 真正的工程价值在通信、监督、可观测性和治理

---

## 一页记住

- 先决定职责，再决定框架
- 消息、任务、状态，是三种不同协作载体
- trace、quality gates、HITL 是生产前提
- 不要为了多 Agent 而多 Agent
- 先证明价值，再扩大规模

如果这五件事做对了，多 Agent 系统就有了上线基础。

---

## 术语速查

- Agent：执行单一职责的智能体
- Orchestrator：负责编排流程的调度者
- Verifier：负责质量检查的 Agent
- HITL：Human-in-the-loop，人工复核
- Trace：系统执行轨迹
- Checkpoint：流程可暂停、可恢复的中间状态

---

## 延伸阅读

- AutoGen 官方文档：https://microsoft.github.io/autogen/stable/
- CrewAI 官方文档：https://docs.crewai.com/
- LangGraph 官方文档：https://docs.langchain.com/langgraph
- LangGraph 持久化与恢复：https://docs.langchain.com/oss/python/langgraph/persistence
- 多 Agent 评估与 RAGAS 相关资料

建议先从“一个小流程”开始，而不是上来就做大而全平台。

---

## Q&A

# 谢谢

欢迎提问、补充和改写你自己的课程项目。

> 结束提示：请同学回想今天最想落地的一个多 Agent 场景。

> 📖 完整讲义已嵌入本节（原 `课程资料.md` / `Marp 幻灯片.md` 已删除）


---

## Part 8 — 小结

### 知识点回顾

| 框架/模式 | 核心抽象 | 适用场景 |
|-----------|---------|----------|
| **AutoGen GraphFlow** | `DiGraphBuilder` + `GraphFlow` | 固定顺序流水线 |
| **AutoGen SelectorGroupChat** | `selector_prompt` | 灵活多轮讨论 |
| **AutoGen Memory** | `ListMemory` | 跨轮次上下文 |
| **CrewAI Sequential** | `Agent + Task + Crew` | 高层级角色协作 |
| **CrewAI Hierarchical** | `Process.hierarchical` | 动态任务分配 |
| **Agentic RAG** | LangGraph `StateGraph` | 检索→生成→验证 |
| **Supervisor + HITL** | `pass_rate` 阈值 | 生产级质量保障 |

### OpenRouter 接入总结

| 框架 | 接入方式 |
|------|----------|
| **AutoGen** | `OpenAIChatCompletionClient(base_url=OPENROUTER_BASE_URL)` |
| **CrewAI** | `LLM(model='openrouter/...', base_url=OPENROUTER_BASE_URL)` |
| **LangChain** | `ChatOpenAI(openai_api_base=OPENROUTER_BASE_URL)` |

### 关键设计原则
1. **单一职责** — 每个 Agent 只做一件事
2. **清晰的 State/Message schema** — 便于调试和重放
3. **终止条件** — 始终设置最大步数/成本上限
4. **质量门** — 关键节点插入 Verifier
5. **成本感知** — 复杂任务用 Sonnet，轻量任务用 Haiku

### 延伸学习
- [OpenRouter 文档](https://openrouter.ai/docs)
- [AutoGen 文档](https://microsoft.github.io/autogen/)
- [CrewAI 文档](https://docs.crewai.com/)
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)
- 本课程 Lecture 32：`agent_supervisor.ipynb`

In [ ]:
print('🎉 完成 Multi-Agent Frameworks 实战 Notebook！')
print()
print('下一步建议：')
print('  1. 完成练习 A/B/C 中的 TODO')
print('  2. 在 OpenRouter 切换不同模型（如 GPT-4o、Gemini）对比效果')
print('  3. 将 Agentic RAG 的模拟知识库替换为真实 ChromaDB')
print('  4. 探索 Lecture 32 的 LangGraph 高级用法')